In [ ]:
import os

base_path = "/content/drive/MyDrive/Colab Notebooks/support_assistant"
docs_path = os.path.join(base_path, "docs")

os.makedirs(docs_path, exist_ok=True)

documents = {
    "doc_01.txt": """Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.""",

    "doc_02.txt": """Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unopened, resalable condition. Approved refunds are credited to the original payment method within 3–5 business days, or instantly to the Zepto wallet if the customer opts for wallet credit. Personal care items that have been opened are non-returnable except in the case of a manufacturing defect. Return pickup, where required, is arranged free of cost by Zepto.""",

    "doc_03.txt": """Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), and Zepto Pass+ (INR 99 per month, free priority delivery, 10% off select categories, and early access to limited-time deals 24 hours before they go live to Basic and Pass members). Membership can be cancelled at any time from account settings; cancelling stops the next billing cycle but does not refund the current membership period.""",

    "doc_04.txt": """Every Zepto order shows a live rider-tracking map from the moment it is packed until delivery, accessible from the 'Track Order' screen. Estimated delivery time updates automatically as the rider moves. If an order's status shows no movement for more than 20 minutes past its original estimated delivery time, customers should contact support directly rather than continue waiting, since this indicates a likely delivery issue.""",

    "doc_05.txt": """Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app, since the rider is dispatched immediately after packing given Zepto's quick-delivery model. If a packed order cannot be delivered due to a Zepto-side issue (for example, rider unavailability), the order is auto-cancelled and fully refunded without any cancellation fee.""",

    "doc_06.txt": """If an order arrives with damaged, spoiled, or missing items, customers must report it within 24 hours of delivery through the 'Report an Issue' button on the order page. Zepto ships a free replacement or issues a full refund for damaged, spoiled, or missing items without requiring the customer to return the original item, unless the order value exceeds INR 1000, in which case a photo of the issue must be submitted through the report form before a replacement or refund is processed.""",

    "doc_07.txt": """Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and INR 1000, and are delivered by email or SMS within minutes of purchase. Gift cards are valid for 1 year from the date of issue and carry no maintenance fees. Gift card balance can be combined with one other payment method at checkout but cannot be combined with another gift card in the same transaction. Gift card balance cannot be redeemed for cash except where required by law.""",

    "doc_08.txt": """Zepto customer support is available via in-app chat 24 hours a day, 7 days a week, given the time-sensitive nature of quick commerce deliveries. Average in-app chat response time is under 2 minutes. Email support is also available for non-urgent queries and is answered within 24 hours on business days. Phone support is not offered."""
}

for filename, content in documents.items():
    filepath = os.path.join(docs_path, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("All 8 documents created successfully!")

print("\nFiles:")
for filename in sorted(os.listdir(docs_path)):
    print(filename)

All 8 documents created successfully!

Files:
doc_01.txt
doc_02.txt
doc_03.txt
doc_04.txt
doc_05.txt
doc_06.txt
doc_07.txt
doc_08.txt


In [ ]:
!pip install -q sentence-transformers chromadb langgraph fastapi uvicorn pydantic

In [ ]:
import sentence_transformers
import chromadb
import langgraph
import fastapi
import pydantic

print("All required libraries imported successfully!")

All required libraries imported successfully!


In [ ]:
import os
import chromadb
from sentence_transformers import SentenceTransformer

base_path = "/content/drive/MyDrive/Colab Notebooks/support_assistant"
docs_path = os.path.join(base_path, "docs")
chroma_path = os.path.join(base_path, "chroma_db")

# Load local embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Create persistent ChromaDB
client = chromadb.PersistentClient(path=chroma_path)

# Create collection
collection = client.get_or_create_collection(
    name="zepto_policies",
    metadata={"hnsw:space": "cosine"}
)

# Load documents
documents = []
ids = []
metadatas = []

for filename in sorted(os.listdir(docs_path)):
    if filename.endswith(".txt"):
        filepath = os.path.join(docs_path, filename)

        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()

        documents.append(text)
        ids.append(filename.replace(".txt", ""))
        metadatas.append({"source": filename})

# Generate embeddings locally
embeddings = model.encode(documents).tolist()

# Store in ChromaDB
collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
)

print("Documents stored successfully!")
print("Number of documents:", collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Documents stored successfully!
Number of documents: 8


In [ ]:
query = "How much does Zepto charge for delivery on orders below INR 149?"

query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=3
)

print("Query:", query)
print("\nRetrieved documents:")

for i, (doc_id, document) in enumerate(
    zip(results["ids"][0], results["documents"][0]), start=1
):
    print(f"\n{i}. {doc_id}")
    print(document[:300])

Query: How much does Zepto charge for delivery on orders below INR 149?

Retrieved documents:

1. doc_01
Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. 

2. doc_03
Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), and Zepto Pass+ (INR 99 per month, free priority delivery, 10% off select categories, and early access to limi

3. doc_08
Zepto customer support is available via in-app chat 24 hours a day, 7 days a week, given the time-sensitive nature of quick commerce deliveries. Average in-app chat response time is under 2 minutes. Email support is also available for non-urgent queries and is answered w

In [ ]:
from typing import TypedDict, List
from pydantic import BaseModel, Field
import os


# -----------------------------
# Final structured response
# -----------------------------
class AnswerResponse(BaseModel):
    answer: str
    sources: List[str]
    confidence: float = Field(ge=0.0, le=1.0)


# -----------------------------
# LangGraph state
# -----------------------------
class SupportState(TypedDict, total=False):
    query: str
    intent: str
    answer: str
    sources: List[str]
    confidence: float


# -----------------------------
# MOCK_LLM configuration
# -----------------------------
MOCK_LLM = os.getenv("MOCK_LLM", "1")

print("MOCK_LLM =", MOCK_LLM)


# -----------------------------
# Intent classification
# -----------------------------
POLICY_KEYWORDS = [
    "delivery",
    "return",
    "refund",
    "membership",
    "tracking",
    "cancel",
    "gift card",
    "support hours"
]


def classify_intent(state: SupportState) -> SupportState:
    query = state["query"].lower()

    if any(keyword in query for keyword in POLICY_KEYWORDS):
        intent = "policy_question"
    else:
        intent = "general_question"

    return {
        **state,
        "intent": intent
    }


print("classify_intent created successfully!")

MOCK_LLM = 1
classify_intent created successfully!


In [ ]:
policy_test = classify_intent({
    "query": "What is the delivery charge?"
})

general_test = classify_intent({
    "query": "What is the capital of India?"
})

print("Policy test:", policy_test)
print("General test:", general_test)

Policy test: {'query': 'What is the delivery charge?', 'intent': 'policy_question'}
General test: {'query': 'What is the capital of India?', 'intent': 'general_question'}


In [ ]:
def retrieve_and_answer(state: SupportState) -> SupportState:
    query = state["query"]

    # Embed the query locally
    query_embedding = model.encode([query]).tolist()

    # Retrieve top 3 chunks using cosine similarity
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=3
    )

    retrieved_ids = results["ids"][0]
    retrieved_docs = results["documents"][0]

    # Required MOCK_LLM baseline
    if MOCK_LLM != "0":
        top_chunk_snippet = retrieved_docs[0][:200]

        answer = (
            f"Based on the retrieved context: "
            f"{top_chunk_snippet}"
        )

        return {
            **state,
            "answer": answer,
            "sources": retrieved_ids,
            "confidence": 1.0
        }

    # Optional real-LLM branch
    # Will be implemented later as an optional extension.
    answer = (
        "Real LLM mode is not enabled in this graded offline baseline."
    )

    return {
        **state,
        "answer": answer,
        "sources": retrieved_ids,
        "confidence": 1.0
    }

In [ ]:
def direct_answer(state: SupportState) -> SupportState:
    if MOCK_LLM != "0":
        answer = "I can only answer questions about Zepto policies right now."

        return {
            **state,
            "answer": answer,
            "sources": [],
            "confidence": 1.0
        }

    # Optional real-LLM branch
    answer = (
        "Real LLM mode is not enabled in this graded offline baseline."
    )

    return {
        **state,
        "answer": answer,
        "sources": [],
        "confidence": 1.0
    }

print("retrieve_and_answer created!")
print("direct_answer created!")

retrieve_and_answer created!
direct_answer created!


In [ ]:
policy_result = retrieve_and_answer({
    "query": "How much is the delivery fee below INR 149?",
    "intent": "policy_question"
})

general_result = direct_answer({
    "query": "What is the capital of India?",
    "intent": "general_question"
})

print("POLICY RESPONSE:")
print(policy_result)

print("\nGENERAL RESPONSE:")
print(general_result)

POLICY RESPONSE:
{'query': 'How much is the delivery fee below INR 149?', 'intent': 'policy_question', 'answer': "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del", 'sources': ['doc_01', 'doc_05', 'doc_03'], 'confidence': 1.0}

GENERAL RESPONSE:
{'query': 'What is the capital of India?', 'intent': 'general_question', 'answer': 'I can only answer questions about Zepto policies right now.', 'sources': [], 'confidence': 1.0}


In [ ]:
from langgraph.graph import StateGraph, START, END


# Create the graph
builder = StateGraph(SupportState)

# Add the 3 required nodes
builder.add_node("classify_intent", classify_intent)
builder.add_node("retrieve_and_answer", retrieve_and_answer)
builder.add_node("direct_answer", direct_answer)


# Start → classify
builder.add_edge(START, "classify_intent")


# Conditional routing
def route_intent(state: SupportState):
    if state["intent"] == "policy_question":
        return "retrieve_and_answer"
    return "direct_answer"


builder.add_conditional_edges(
    "classify_intent",
    route_intent,
    {
        "retrieve_and_answer": "retrieve_and_answer",
        "direct_answer": "direct_answer"
    }
)


# Both answer nodes → END
builder.add_edge("retrieve_and_answer", END)
builder.add_edge("direct_answer", END)


# Compile graph
graph = builder.compile()

print("LangGraph compiled successfully!")

LangGraph compiled successfully!


In [ ]:
policy_query = {
    "query": "How much is the delivery fee below INR 149?"
}

policy_response = graph.invoke(policy_query)

print("POLICY QUERY")
print(policy_response)

POLICY QUERY
{'query': 'How much is the delivery fee below INR 149?', 'intent': 'policy_question', 'answer': "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del", 'sources': ['doc_01', 'doc_05', 'doc_03'], 'confidence': 1.0}


In [ ]:
general_query = {
    "query": "What is the capital of India?"
}

general_response = graph.invoke(general_query)

print("\nGENERAL QUERY")
print(general_response)


GENERAL QUERY
{'query': 'What is the capital of India?', 'intent': 'general_question', 'answer': 'I can only answer questions about Zepto policies right now.', 'sources': [], 'confidence': 1.0}


In [ ]:
def validate_response(state: SupportState) -> AnswerResponse:
    response = AnswerResponse(
        answer=state["answer"],
        sources=state.get("sources", []),
        confidence=state.get("confidence", 1.0)
    )

    return response


# Test policy response
validated_policy = validate_response(policy_response)

print("POLICY JSON:")
print(validated_policy.model_dump_json(indent=2))


# Test general response
validated_general = validate_response(general_response)

print("\nGENERAL JSON:")
print(validated_general.model_dump_json(indent=2))

POLICY JSON:
{
  "answer": "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del",
  "sources": [
    "doc_01",
    "doc_05",
    "doc_03"
  ],
  "confidence": 1.0
}

GENERAL JSON:
{
  "answer": "I can only answer questions about Zepto policies right now.",
  "sources": [],
  "confidence": 1.0
}


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel


# Request schema
class AskRequest(BaseModel):
    query: str


# FastAPI application
app = FastAPI(
    title="Zepto Support Assistant",
    description="Offline RAG-based Zepto policy support assistant",
    version="1.0.0"
)


@app.post("/ask", response_model=AnswerResponse)
def ask(request: AskRequest):
    # Run the LangGraph
    result = graph.invoke({
        "query": request.query
    })

    # Validate the final response
    validated = AnswerResponse(
        answer=result["answer"],
        sources=result.get("sources", []),
        confidence=result.get("confidence", 1.0)
    )

    return validated


print("FastAPI application created successfully!")

FastAPI application created successfully!


In [ ]:
import os

base_path = "/content/drive/MyDrive/Colab Notebooks/support_assistant"

main_code = r'''
import os
from typing import TypedDict, List

import chromadb
from sentence_transformers import SentenceTransformer
from fastapi import FastAPI
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END


# ============================================================
# Configuration
# ============================================================

BASE_PATH = os.path.dirname(os.path.abspath(__file__))
CHROMA_PATH = os.path.join(BASE_PATH, "chroma_db")

MOCK_LLM = os.getenv("MOCK_LLM", "1")


# ============================================================
# Embedding + ChromaDB
# ============================================================

model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = client.get_or_create_collection(
    name="zepto_policies",
    metadata={"hnsw:space": "cosine"}
)


# ============================================================
# Pydantic schemas
# ============================================================

class AskRequest(BaseModel):
    query: str


class AnswerResponse(BaseModel):
    answer: str
    sources: List[str]
    confidence: float = Field(ge=0.0, le=1.0)


# ============================================================
# LangGraph state
# ============================================================

class SupportState(TypedDict, total=False):
    query: str
    intent: str
    answer: str
    sources: List[str]
    confidence: float


# ============================================================
# Intent classification
# ============================================================

POLICY_KEYWORDS = [
    "delivery",
    "return",
    "refund",
    "membership",
    "tracking",
    "cancel",
    "gift card",
    "support hours"
]


def classify_intent(state: SupportState) -> SupportState:
    query = state["query"].lower()

    if any(keyword in query for keyword in POLICY_KEYWORDS):
        intent = "policy_question"
    else:
        intent = "general_question"

    return {
        **state,
        "intent": intent
    }


# ============================================================
# Retrieval + answer
# ============================================================

def retrieve_and_answer(state: SupportState) -> SupportState:
    query = state["query"]

    query_embedding = model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=3
    )

    retrieved_ids = results["ids"][0]
    retrieved_docs = results["documents"][0]

    if MOCK_LLM != "0":
        top_chunk_snippet = retrieved_docs[0][:200]

        answer = (
            f"Based on the retrieved context: "
            f"{top_chunk_snippet}"
        )

        return {
            **state,
            "answer": answer,
            "sources": retrieved_ids,
            "confidence": 1.0
        }

    # Optional real-LLM branch.
    # The graded baseline uses MOCK_LLM=1.
    answer = (
        "Real LLM mode is optional and is not enabled "
        "in the offline graded baseline."
    )

    return {
        **state,
        "answer": answer,
        "sources": retrieved_ids,
        "confidence": 1.0
    }


# ============================================================
# Direct answer
# ============================================================

def direct_answer(state: SupportState) -> SupportState:

    if MOCK_LLM != "0":
        answer = (
            "I can only answer questions about Zepto policies right now."
        )

        return {
            **state,
            "answer": answer,
            "sources": [],
            "confidence": 1.0
        }

    # Optional real-LLM branch.
    answer = (
        "Real LLM mode is optional and is not enabled "
        "in the offline graded baseline."
    )

    return {
        **state,
        "answer": answer,
        "sources": [],
        "confidence": 1.0
    }


# ============================================================
# LangGraph
# ============================================================

builder = StateGraph(SupportState)

builder.add_node("classify_intent", classify_intent)
builder.add_node("retrieve_and_answer", retrieve_and_answer)
builder.add_node("direct_answer", direct_answer)

builder.add_edge(START, "classify_intent")


def route_intent(state: SupportState):
    if state["intent"] == "policy_question":
        return "retrieve_and_answer"

    return "direct_answer"


builder.add_conditional_edges(
    "classify_intent",
    route_intent,
    {
        "retrieve_and_answer": "retrieve_and_answer",
        "direct_answer": "direct_answer"
    }
)

builder.add_edge("retrieve_and_answer", END)
builder.add_edge("direct_answer", END)

graph = builder.compile()


# ============================================================
# FastAPI
# ============================================================

app = FastAPI(
    title="Zepto Support Assistant",
    description="Offline RAG-based Zepto policy support assistant",
    version="1.0.0"
)


@app.post("/ask", response_model=AnswerResponse)
def ask(request: AskRequest):

    result = graph.invoke({
        "query": request.query
    })

    return AnswerResponse(
        answer=result["answer"],
        sources=result.get("sources", []),
        confidence=result.get("confidence", 1.0)
    )
'''

main_path = os.path.join(base_path, "main.py")

with open(main_path, "w", encoding="utf-8") as f:
    f.write(main_code)

print("main.py created successfully!")
print(main_path)

main.py created successfully!
/content/drive/MyDrive/Colab Notebooks/support_assistant/main.py


In [ ]:
%cd "/content/drive/MyDrive/Colab Notebooks/support_assistant"
!uvicorn main:app --host 0.0.0.0 --port 8000

/content/drive/MyDrive/Colab Notebooks/support_assistant
Loading weights: 100% 103/103 [00:00<00:00, 9872.11it/s]
INFO:     Started server process [17809]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [ ]:
%cd "/content/drive/MyDrive/Colab Notebooks/support_assistant"

!python -m uvicorn main:app --host 127.0.0.1 --port 8000 &

/content/drive/MyDrive/Colab Notebooks/support_assistant
Loading weights: 100% 103/103 [00:00<00:00, 6568.25it/s]
INFO:     Started server process [18014]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [ ]:
import subprocess
import os

os.chdir("/content/drive/MyDrive/Colab Notebooks/support_assistant")

process = subprocess.Popen(
    ["python", "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print("FastAPI started in background")
print("PID:", process.pid)

FastAPI started in background
PID: 18130


In [ ]:
import requests

url = "http://127.0.0.1:8000/ask"

response = requests.post(
    url,
    json={
        "query": "How much does Zepto charge for delivery below INR 149?"
    }
)

print("Status:", response.status_code)
print("Response:")
print(response.json())

Status: 200
Response:
{'answer': "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del", 'sources': ['doc_01', 'doc_03', 'doc_08'], 'confidence': 1.0}


In [ ]:
response = requests.post(
    url,
    json={"query": "What is the capital of India?"}
)

print(response.status_code)
print(response.json())

200
{'answer': 'I can only answer questions about Zepto policies right now.', 'sources': [], 'confidence': 1.0}


In [ ]:
import os

base_path = "/content/drive/MyDrive/Colab Notebooks/support_assistant"

prompt_code = '''
STRUCTURED_PROMPT_TEMPLATE = """
ROLE:
You are Zepto's customer support assistant. Answer questions using only the
provided Zepto policy context.

CONTEXT:
{context}

TASK:
Answer the customer's question accurately using the provided context.

FORMAT:
Return a JSON object with exactly these fields:
- answer: string
- sources: list of document or chunk IDs
- confidence: number between 0 and 1

LENGTH:
Keep the answer concise and directly relevant to the customer's question.

NEGATIVE CONSTRAINT:
Do not answer using information that is not present in the provided context.
Do not invent or assume Zepto policies. If the context does not contain the
answer, clearly say that the provided context does not contain enough
information.

FEW-SHOT EXAMPLE:
Question:
"What is the delivery fee for an order below INR 149?"

Context:
"Orders below INR 149 incur a flat INR 25 delivery fee."

Expected answer:
{
  "answer": "Orders below INR 149 incur a flat INR 25 delivery fee.",
  "sources": ["doc_01"],
  "confidence": 1.0
}

Now answer the customer's question using only the provided context.
"""
'''
prompt_path = os.path.join(base_path, "prompts.py")

with open(prompt_path, "w", encoding="utf-8") as f:
    f.write(prompt_code)

print("prompts.py created successfully!")

prompts.py created successfully!


In [ ]:
base_path = "/content/drive/MyDrive/Colab Notebooks/support_assistant"

requirements = """sentence-transformers
chromadb
langgraph
fastapi
uvicorn
pydantic
joblib
"""

with open(f"{base_path}/requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt created successfully!")

requirements.txt created successfully!


In [ ]:
dockerfile = """FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY main.py .
COPY prompts.py .
COPY docs ./docs
COPY chroma_db ./chroma_db

EXPOSE 7860

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"]
"""

with open(f"{base_path}/Dockerfile", "w") as f:
    f.write(dockerfile)

print("Dockerfile created successfully!")

Dockerfile created successfully!


In [ ]:
import os

base_path = "/content/drive/MyDrive/Colab Notebooks/support_assistant"

readme = """# Zepto Support Assistant

## Module 3

This project implements an offline RAG-based Zepto Support Assistant using
local embeddings, ChromaDB, LangGraph, Pydantic and FastAPI.

## Architecture

Policy Documents
    |
    v
Document Ingestion
    |
    v
all-MiniLM-L6-v2 Embeddings
    |
    v
ChromaDB - zepto_policies
    |
    v
LangGraph Intent Router
    |
    +--> policy_question --> retrieve_and_answer
    |
    +--> general_question --> direct_answer
    |
    v
Pydantic Structured Response
    |
    v
FastAPI /ask

## Documents

Eight Zepto policy documents are stored in the docs directory.

The documents cover:

1. Delivery Policy
2. Returns and Refunds
3. Membership Tiers
4. Order Tracking
5. Order Cancellation
6. Damaged or Missing Items
7. Gift Cards
8. Customer Support Hours

## Embeddings

The documents are embedded locally using:

all-MiniLM-L6-v2

The embeddings are stored in the ChromaDB collection:

zepto_policies

## Retrieval

For policy questions, the retrieve_and_answer node embeds the query and
retrieves the top 3 most similar documents using cosine similarity.

## Intent Classification

The classify_intent node uses the required keyword heuristic in the default
MOCK_LLM mode.

Policy keywords include delivery, return, refund, membership, tracking,
cancel, gift card and support hours.

## Mock LLM Mode

The default mode is MOCK_LLM=1.

No external LLM API call is required.

For policy questions, the system returns an answer based on the top retrieved
document.

For general questions, the system returns:

I can only answer questions about Zepto policies right now.

## Structured Output

The final response is validated using Pydantic.

Example:

{
  "answer": "Based on the retrieved context: ...",
  "sources": ["doc_01", "doc_03", "doc_08"],
  "confidence": 1.0
}

For general questions, sources is an empty list.

## Example 1 - Policy Question

Request:

{
  "query": "How much does Zepto charge for delivery below INR 149?"
}

Response:

{
  "answer": "Based on the retrieved context: ...",
  "sources": ["doc_01", "doc_03", "doc_08"],
  "confidence": 1.0
}

This query is routed to policy_question and then to retrieve_and_answer.

## Example 2 - General Question

Request:

{
  "query": "What is the capital of India?"
}

Response:

{
  "answer": "I can only answer questions about Zepto policies right now.",
  "sources": [],
  "confidence": 1.0
}

This query is routed to general_question and then to direct_answer.

## FastAPI

Run locally with:

uvicorn main:app --host 0.0.0.0 --port 8000

Endpoint:

POST /ask

## Docker

Build:

docker build -t zepto-support-assistant .

Run:

docker run -p 7860:7860 zepto-support-assistant

The application listens on port 7860 inside the container.

## Project Files

docs/
main.py
prompts.py
requirements.txt
Dockerfile
README.md
chroma_db/

## Conclusion

The system provides a deterministic offline RAG baseline for Zepto policy
questions. Documents are embedded locally, indexed in ChromaDB, routed using
LangGraph and returned through a validated FastAPI response.
"""

readme_path = os.path.join(base_path, "README.md")

with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme)

print("README.md created successfully!")

README.md created successfully!


In [ ]:
import os

base_path = "/content/drive/MyDrive/Colab Notebooks/support_assistant"

for root, dirs, files in os.walk(base_path):
    level = root.replace(base_path, "").count(os.sep)
    indent = "  " * level
    print(indent + os.path.basename(root) + "/")

    for file in sorted(files):
        print(indent + "  " + file)

support_assistant/
  Dockerfile
  README.md
  llm_validation.py
  main.py
  prompts.py
  requirements.txt
  chroma_db/
    chroma.sqlite3
    .ipynb_checkpoints/
    870bfef9-d2ed-48fb-a019-42ab2e86dc47/
      data_level0.bin
      header.bin
      length.bin
      link_lists.bin
  __pycache__/
    main.cpython-312.pyc
  docs/
    doc_01.txt
    doc_02.txt
    doc_03.txt
    doc_04.txt
    doc_05.txt
    doc_06.txt
    doc_07.txt
    doc_08.txt


In [ ]:
import os

base_path = "/content/drive/MyDrive/Colab Notebooks/support_assistant"

prompt_code = '''
STRUCTURED_PROMPT_TEMPLATE = """
ROLE:
You are Zepto's customer support assistant.

CONTEXT:
{context}

TASK:
Answer the customer's question using only the provided Zepto policy context.

FORMAT:
Return valid JSON with exactly these fields:
answer (string),
sources (list of document IDs),
confidence (float from 0 to 1).

LENGTH:
Keep the answer concise and directly relevant.

NEGATIVE CONSTRAINT:
Do not use information that is not present in the provided context.
Do not invent or assume Zepto policies.

FEW-SHOT EXAMPLE:

Question:
How much is delivery below INR 149?

Context:
Orders below INR 149 incur a flat INR 25 delivery fee.

Expected JSON:
{
  "answer": "Orders below INR 149 incur a flat INR 25 delivery fee.",
  "sources": ["doc_01"],
  "confidence": 1.0
}

Now answer the customer's question using only the provided context.
"""
'''

with open(os.path.join(base_path, "prompts.py"), "w", encoding="utf-8") as f:
    f.write(prompt_code)

print("prompts.py updated successfully!")

prompts.py updated successfully!


In [ ]:
retry_code = '''
from pydantic import ValidationError
from prompts import STRUCTURED_PROMPT_TEMPLATE


def validate_llm_response(raw_output):
    """
    Validate the LLM JSON response.

    If validation fails, the real-LLM path can retry up to
    two additional times with a corrective instruction.
    """

    from main import AnswerResponse

    try:
        return AnswerResponse.model_validate_json(raw_output)

    except ValidationError as first_error:

        corrective_instruction = (
            "Your previous response was invalid. "
            "Return ONLY valid JSON with exactly these fields: "
            "answer, sources, confidence. "
            "confidence must be between 0 and 1."
        )

        for attempt in range(2):
            # Placeholder for a real LLM call.
            # In MOCK_LLM=1 this function is never used.
            #
            # A real implementation would send:
            # STRUCTURED_PROMPT_TEMPLATE + corrective_instruction
            # to the selected LLM here.
            #
            # The returned raw JSON should then be validated again.

            pass

        return {
            "error": "LLM output failed schema validation after 2 retries",
            "details": str(first_error)
        }
'''

with open(os.path.join(base_path, "llm_validation.py"), "w", encoding="utf-8") as f:
    f.write(retry_code)

print("llm_validation.py created successfully!")

llm_validation.py created successfully!


In [ ]:
base_path = "/content/drive/MyDrive/Colab Notebooks/support_assistant"

with open(f"{base_path}/requirements.txt", "r") as f:
    print(f.read())

sentence-transformers
chromadb
langgraph
fastapi
uvicorn
pydantic
joblib



In [ ]:
with open(f"{base_path}/Dockerfile", "r") as f:
    print(f.read())

FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY main.py .
COPY prompts.py .
COPY docs ./docs
COPY chroma_db ./chroma_db

EXPOSE 7860

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"]



In [ ]:
import os

for item in sorted(os.listdir(base_path)):
    print(item)

Dockerfile
README.md
__pycache__
chroma_db
docs
llm_validation.py
main.py
prompts.py
requirements.txt


In [ ]:
from pathlib import Path
import shutil

# FINAL ASSIGNMENT LOCATION
FINAL = Path("/content/drive/MyDrive/zepto-assignment/support_assistant")

# Create required folders
(FINAL / "docs").mkdir(parents=True, exist_ok=True)
(FINAL / "chroma_db").mkdir(parents=True, exist_ok=True)

# Existing support_assistant files location
SOURCE = Path("/content/drive/MyDrive/Colab Notebooks/support_assistant")

# Separate files that must be directly inside support_assistant
files = [
    "main.py",
    "prompts.py",
    "llm_validation.py",
    "Dockerfile",
    "requirements.txt",
    "README.md"
]

for filename in files:
    src = SOURCE / filename
    dst = FINAL / filename

    if src.exists():
        shutil.copy2(src, dst)
        print("✅", filename)
    else:
        print("⚠️ Not found:", filename)

print("\nFINAL SUPPORT ASSISTANT:")
for item in sorted(FINAL.iterdir()):
    print("📁" if item.is_dir() else "📄", item.name)

✅ main.py
✅ prompts.py
✅ llm_validation.py
✅ Dockerfile
✅ requirements.txt
✅ README.md

FINAL SUPPORT ASSISTANT:
📄 Dockerfile
📄 README.md
📁 __pycache__
📁 chroma_db
📁 docs
📄 llm_validation.py
📄 main.py
📄 prompts.py
📄 requirements.txt


In [ ]:
import sys

sys.path.insert(
    0,
    "/content/drive/MyDrive/zepto-assignment/support_assistant"
)

import main

print("✅ main.py imported successfully")
print("FastAPI app exists:", hasattr(main, "app"))

✅ main.py imported successfully
FastAPI app exists: True


In [ ]:
from fastapi.testclient import TestClient
import main

client = TestClient(main.app)

response = client.post(
    "/ask",
    json={"query": "How much does Zepto charge for delivery on orders below INR 149?"}
)

print("Status:", response.status_code)
print("Response:", response.json())
import traceback
import main
from fastapi.testclient import TestClient

client = TestClient(main.app)

try:
    response = client.post(
        "/ask",
        json={"query": "How much does Zepto charge for delivery on orders below INR 149?"}
    )
    print("Status:", response.status_code)
    print(response.json())

except Exception:
    traceback.print_exc()


Status: 200
Response: {'answer': "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del", 'sources': ['doc_01', 'doc_03', 'doc_07'], 'confidence': 1.0}
Status: 200
{'answer': "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del", 'sources': ['doc_01', 'doc_03', 'doc_07'], 'confidence': 1.0}


In [ ]:
import traceback
import main
from fastapi.testclient import TestClient

client = TestClient(main.app)

try:
    response = client.post(
        "/ask",
        json={"query": "How much does Zepto charge for delivery on orders below INR 149?"}
    )
    print("Status:", response.status_code)
    print(response.json())

except Exception:
    traceback.print_exc()

Status: 200
{'answer': "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del", 'sources': ['doc_01', 'doc_03', 'doc_07'], 'confidence': 1.0}


In [ ]:
try:
    response = client.post(
        "/ask",
        json={"query": "How much does Zepto charge for delivery on orders below INR 149?"}
    )
    print(response.status_code)
    print(response.json())

except Exception as e:
    print("ERROR TYPE:", type(e).__name__)
    print("ERROR:", str(e))

200
{'answer': "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del", 'sources': ['doc_01', 'doc_03', 'doc_07'], 'confidence': 1.0}


In [ ]:
import chromadb

CHROMA_PATH = "/content/drive/MyDrive/zepto-assignment/support_assistant/chroma_db"

client = chromadb.PersistentClient(path=CHROMA_PATH)

print("Collections:", client.list_collections())

collection = client.get_collection("zepto_policies")

print("Document count:", collection.count())

result = collection.query(
    query_texts=["How much is delivery below INR 149?"],
    n_results=3
)

print("Retrieved documents:", result["documents"])

Collections: [Collection(name=zepto_policies)]
Document count: 8
Retrieved documents: [["Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee.", 'Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), and Zepto Pass+ (INR 99 per month, free priority delivery, 10% off select categories, and early access to limited-time deals).', 'Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and INR 1000, and are delivered by email or SMS within minutes of purchase. Gift cards are valid for 1 year from the date of issue and carry no maintenance fees.']]


In [ ]:
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer

BASE = Path("/content/drive/MyDrive/zepto-assignment/support_assistant")
DOCS = BASE / "docs"
CHROMA = BASE / "chroma_db"

# Read the actual documents from final docs folder
doc_files = sorted(DOCS.glob("doc_*.txt"))

print("Documents found:", len(doc_files))

if len(doc_files) != 8:
    print("⚠️ Expected 8 documents, but found:", len(doc_files))
else:
    texts = [f.read_text(encoding="utf-8") for f in doc_files]
    ids = [f.stem for f in doc_files]

    model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = model.encode(texts).tolist()

    client = chromadb.PersistentClient(path=str(CHROMA))

    collection = client.get_or_create_collection(
        name="zepto_policies",
        metadata={"hnsw:space": "cosine"}
    )

    # Remove old records if any
    if collection.count() > 0:
        collection.delete(ids=collection.get()["ids"])

    collection.add(
        ids=ids,
        documents=texts,
        embeddings=embeddings
    )

    print("✅ Documents stored successfully!")
    print("Number of documents:", collection.count())
    print("ChromaDB:", CHROMA)

Documents found: 8


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Documents stored successfully!
Number of documents: 8
ChromaDB: /content/drive/MyDrive/zepto-assignment/support_assistant/chroma_db


In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/zepto-assignment/support_assistant")
DOCS = BASE / "docs"

DOCS.mkdir(parents=True, exist_ok=True)

documents = {
    "doc_01.txt": """Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee.""",

    "doc_02.txt": """Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unopened, resalable condition. Approved refunds are credited to the original payment method within 3–5 business days, or instantly to the Zepto wallet if the customer opts for wallet credit.""",

    "doc_03.txt": """Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), and Zepto Pass+ (INR 99 per month, free priority delivery, 10% off select categories, and early access to limited-time deals).""",

    "doc_04.txt": """Every Zepto order shows a live rider-tracking map from the moment it is packed until delivery, accessible from the Track Order screen. Estimated delivery time updates automatically as the rider moves. If an order's status shows no movement for more than 20 minutes past its original estimated delivery time, customers should contact support.""",

    "doc_05.txt": """Orders can be cancelled free of cost any time before the order status changes to Packed, typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app.""",

    "doc_06.txt": """If an order arrives with damaged, spoiled, or missing items, customers must report it within 24 hours of delivery through the Report an Issue button on the order page. Zepto ships a free replacement or issues a full refund for damaged, spoiled, or missing items.""",

    "doc_07.txt": """Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and INR 1000, and are delivered by email or SMS within minutes of purchase. Gift cards are valid for 1 year from the date of issue and carry no maintenance fees.""",

    "doc_08.txt": """Zepto customer support is available via in-app chat 24 hours a day, 7 days a week. Average in-app chat response time is under 2 minutes. Email support is also available for non-urgent queries and is answered within 24 hours on business days. Phone support is not offered."""
}

for filename, text in documents.items():
    (DOCS / filename).write_text(text, encoding="utf-8")

print("✅ Documents created:", len(list(DOCS.glob("doc_*.txt"))))
print("Location:", DOCS)

for f in sorted(DOCS.glob("doc_*.txt")):
    print("📄", f.name)

✅ Documents created: 8
Location: /content/drive/MyDrive/zepto-assignment/support_assistant/docs
📄 doc_01.txt
📄 doc_02.txt
📄 doc_03.txt
📄 doc_04.txt
📄 doc_05.txt
📄 doc_06.txt
📄 doc_07.txt
📄 doc_08.txt


In [ ]:
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer

BASE = Path("/content/drive/MyDrive/zepto-assignment/support_assistant")
DOCS = BASE / "docs"
CHROMA = BASE / "chroma_db"

doc_files = sorted(DOCS.glob("doc_*.txt"))

print("Documents found:", len(doc_files))

texts = [f.read_text(encoding="utf-8") for f in doc_files]
ids = [f.stem for f in doc_files]

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(texts).tolist()

client = chromadb.PersistentClient(path=str(CHROMA))

collection = client.get_or_create_collection(
    name="zepto_policies",
    metadata={"hnsw:space": "cosine"}
)

# Clear old empty/incorrect records
if collection.count() > 0:
    collection.delete(ids=collection.get()["ids"])

collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings
)

print("✅ Documents indexed successfully!")
print("Number of documents:", collection.count())
print("ChromaDB location:", CHROMA)

Documents found: 8


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Documents indexed successfully!
Number of documents: 8
ChromaDB location: /content/drive/MyDrive/zepto-assignment/support_assistant/chroma_db


In [ ]:
import chromadb

CHROMA_PATH = "/content/drive/MyDrive/zepto-assignment/support_assistant/chroma_db"

client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_collection("zepto_policies")

result = collection.query(
    query_texts=["How much does Zepto charge for orders below INR 149?"],
    n_results=3
)

print("Documents retrieved:", len(result["documents"][0]))

for i, doc in enumerate(result["documents"][0], 1):
    print(f"\n--- Result {i} ---")
    print(doc[:250])

Documents retrieved: 3

--- Result 1 ---
Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), and Zepto Pass+ (INR 99 per month, free priority delivery, 1

--- Result 2 ---
Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below

--- Result 3 ---
Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and INR 1000, and are delivered by email or SMS within minutes of purchase. Gift cards are valid for 1 year from the date of issue and carry no maintenance fees.


In [ ]:
from fastapi.testclient import TestClient
import main

client = TestClient(main.app)

response = client.post(
    "/ask",
    json={"query": "How much does Zepto charge for orders below INR 149?"}
)

print("STATUS:", response.status_code)
print("RESPONSE:")
print(response.json())

STATUS: 200
RESPONSE:
{'answer': 'I can only answer questions about Zepto policies right now.', 'sources': [], 'confidence': 1.0}


In [ ]:
from pathlib import Path

path = Path("/content/drive/MyDrive/zepto-assignment/support_assistant")

for item in sorted(path.iterdir()):
    print("📁" if item.is_dir() else "📄", item.name)

📄 Dockerfile
📄 README.md
📁 __pycache__
📁 chroma_db
📁 docs
📄 llm_validation.py
📄 main.py
📄 prompts.py
📄 requirements.txt


In [ ]:
from pathlib import Path

base = Path("/content/drive/MyDrive/zepto-assignment")

print("ZEPTО ASSIGNMENT:")
print(base)

for item in sorted(base.iterdir()):
    print("📁" if item.is_dir() else "📄", item.name)

print("\nSUPPORT ASSISTANT:")
support = base / "support_assistant"

for item in sorted(support.iterdir()):
    print("📁" if item.is_dir() else "📄", item.name)

ZEPTО ASSIGNMENT:
/content/drive/MyDrive/zepto-assignment
📁 support_assistant

SUPPORT ASSISTANT:
📄 Dockerfile
📄 README.md
📁 __pycache__
📁 chroma_db
📁 docs
📄 llm_validation.py
📄 main.py
📄 prompts.py
📄 requirements.txt


In [91]:
from pathlib import Path
import json

notebook = Path("/content/drive/MyDrive/zepto-assignment/support_assistant/support_assistant.ipynb")

print("Notebook exists:", notebook.exists())

Notebook exists: False


In [96]:
from pathlib import Path

root = Path("/content/drive/MyDrive")

matches = list(root.rglob("support_assistant.ipynb"))

print("Found:", len(matches))

for p in matches:
    print(p)

Found: 0


In [97]:
from google.colab import drive

drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [99]:
import os

print(os.path.ismount("/content/drive"))

False


In [100]:
from pathlib import Path

for base in [Path("/content"), Path("/content/drive")]:
    if base.exists():
        print(f"\nSearching in {base} ...")
        for p in base.rglob("main.py"):
            print(p)


Searching in /content ...
/content/drive/MyDrive/zepto-assignment/support_assistant/main.py
/content/drive/MyDrive/Colab Notebooks/support_assistant/main.py

Searching in /content/drive ...
/content/drive/MyDrive/zepto-assignment/support_assistant/main.py
/content/drive/MyDrive/Colab Notebooks/support_assistant/main.py


In [101]:
from pathlib import Path

folder = Path("/content/drive/MyDrive/zepto-assignment/support_assistant")

print("FINAL SUPPORT ASSISTANT FILES:")
for item in sorted(folder.iterdir()):
    print("📁" if item.is_dir() else "📄", item.name)

FINAL SUPPORT ASSISTANT FILES:
📄 Dockerfile
📄 README.md
📁 __pycache__
📁 chroma_db
📁 docs
📄 llm_validation.py
📄 main.py
📄 prompts.py
📄 requirements.txt


In [112]:
from google.colab import files

files.download("/content/drive/MyDrive/zepto-assignment/support_assistant/main.py")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [105]:
files.download("/content/drive/MyDrive/zepto-assignment/support_assistant/prompts.py")
files.download("/content/drive/MyDrive/zepto-assignment/support_assistant/llm_validation.py")
files.download("/content/drive/MyDrive/zepto-assignment/support_assistant/requirements.txt")
files.download("/content/drive/MyDrive/zepto-assignment/support_assistant/Dockerfile")
files.download("/content/drive/MyDrive/zepto-assignment/support_assistant/README.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [106]:
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer

# FINAL SUPPORT ASSISTANT FOLDER
BASE = Path("/content/drive/MyDrive/zepto-assignment/support_assistant")

DOCS = BASE / "docs"
CHROMA = BASE / "chroma_db"

DOCS.mkdir(parents=True, exist_ok=True)
CHROMA.mkdir(parents=True, exist_ok=True)

# 8 required corpus documents
documents = {
    "doc_01.txt": """Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee.""",

    "doc_02.txt": """Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect. Non-perishable packaged items may be returned within 7 days of delivery in unopened, resalable condition. Approved refunds are credited to the original payment method within 3–5 business days.""",

    "doc_03.txt": """Zepto offers three account tiers: Basic (free, default tier), Zepto Pass (INR 49 per month), and Zepto Pass+ (INR 99 per month). Membership can be cancelled at any time from account settings.""",

    "doc_04.txt": """Every Zepto order shows a live rider-tracking map from the moment it is packed until delivery, accessible from the Track Order screen. If an order's status shows no movement for more than 20 minutes past its original estimated delivery time, customers should contact support.""",

    "doc_05.txt": """Orders can be cancelled free of cost any time before the order status changes to Packed, typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app.""",

    "doc_06.txt": """If an order arrives with damaged, spoiled, or missing items, customers must report it within 24 hours of delivery through the Report an Issue button on the order page. Zepto ships a free replacement or issues a full refund for damaged, spoiled, or missing items.""",

    "doc_07.txt": """Zepto gift cards are available in denominations of INR 100, INR 250, INR 500, and INR 1000. Gift cards are valid for 1 year from the date of issue and carry no maintenance fees.""",

    "doc_08.txt": """Zepto customer support is available via in-app chat 24 hours a day, 7 days a week. Email support is available for non-urgent queries and is answered within 24 hours on business days. Phone support is not offered."""
}

# Save docs as separate .txt files
for filename, text in documents.items():
    (DOCS / filename).write_text(text, encoding="utf-8")

print("✅ Docs created:", len(list(DOCS.glob("*.txt"))))

# Create ChromaDB
model = SentenceTransformer("all-MiniLM-L6-v2")

ids = [Path(name).stem for name in documents]
texts = list(documents.values())
embeddings = model.encode(texts).tolist()

client = chromadb.PersistentClient(path=str(CHROMA))

collection = client.get_or_create_collection(
    name="zepto_policies",
    metadata={"hnsw:space": "cosine"}
)

# Clear previous records
if collection.count() > 0:
    collection.delete(ids=collection.get()["ids"])

collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings
)

print("✅ ChromaDB created")
print("✅ Number of documents:", collection.count())
print("📍 Location:", CHROMA)

✅ Docs created: 8


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ ChromaDB created
✅ Number of documents: 8
📍 Location: /content/drive/MyDrive/zepto-assignment/support_assistant/chroma_db


In [113]:
files.download("/content/drive/MyDrive/zepto-assignment/support_assistant/chroma_db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>